<a href="https://colab.research.google.com/github/MendAmar555/LAB-3/blob/lab3-2/lab3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%%writefile saxpy.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <chrono>

// ── GPU kernel (өөрчлөгдөөгүй) ──────────────────────────────────────────────
__global__ void saxpy_kernel(int N, float alpha, float *x, float *y, float *result)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    if (index < N) result[index] = alpha * x[index] + y[index];
}

// ── Нэмэлт: хугацаа + bandwidth хэмжих бүтэц ─────────────────────────────
struct PerfResult {
    float h2d_ms;           // Host → Device хугацаа
    float kernel_ms;        // GPU kernel хугацаа
    float d2h_ms;           // Device → Host хугацаа
    float total_ms;         // Нийт хугацаа (malloc орно)
    float h2d_bandwidth_gb; // H2D bandwidth (GB/s)
    float d2h_bandwidth_gb; // D2H bandwidth (GB/s)
    float kernel_bandwidth_gb; // Kernel effective bandwidth (GB/s)
};

PerfResult run_saxpy(int N, float alpha, float *host_x, float *host_y, float *host_result)
{
    int size = N * sizeof(float);
    float *device_x, *device_y, *device_result;

    // ── Нийт хугацааны CPU цаг ──────────────────────────────────────────────
    auto totalStart = std::chrono::high_resolution_clock::now();

    cudaMalloc(&device_x,      size);
    cudaMalloc(&device_y,      size);
    cudaMalloc(&device_result, size);

    // ── CUDA Event-үүд (H2D, Kernel, D2H тус бүрд) ──────────────────────────
    cudaEvent_t h2d_start, h2d_stop;

    cudaEvent_t k_start,   k_stop;

    cudaEvent_t d2h_start, d2h_stop;


    cudaEventCreate(&h2d_start);  cudaEventCreate(&h2d_stop);

    cudaEventCreate(&k_start);    cudaEventCreate(&k_stop);

    cudaEventCreate(&d2h_start);  cudaEventCreate(&d2h_stop);


    // ── H2D хугацаа хэмжих ─────────────────────────────────────────────────
    cudaEventRecord(h2d_start);

    cudaMemcpy(device_x, host_x, size, cudaMemcpyHostToDevice);
    cudaMemcpy(device_y, host_y, size, cudaMemcpyHostToDevice);
    cudaEventRecord(h2d_stop);

    cudaEventSynchronize(h2d_stop);


    // ── Thread тохиргоо ────────────────────────────────────────────────────
    const int threadsPerBlock = 256;
    const int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // ── Kernel хугацаа хэмжих ─────────────────────────────────────────────
    cudaEventRecord(k_start);

    saxpy_kernel<<<blocks, threadsPerBlock>>>(N, alpha, device_x, device_y, device_result);
    cudaEventRecord(k_stop);

    cudaEventSynchronize(k_stop);


    // ── D2H хугацаа хэмжих ─────────────────────────────────────────────────
    cudaEventRecord(d2h_start);

    cudaMemcpy(host_result, device_result, size, cudaMemcpyDeviceToHost);
    cudaEventRecord(d2h_stop);

    cudaEventSynchronize(d2h_stop);


    auto totalEnd = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> totalMs = totalEnd - totalStart;

    // ── Хугацаа унших ──────────────────────────────────────────────────────
    PerfResult r;

    cudaEventElapsedTime(&r.h2d_ms,    h2d_start, h2d_stop);

    cudaEventElapsedTime(&r.kernel_ms, k_start,   k_stop);

    cudaEventElapsedTime(&r.d2h_ms,    d2h_start, d2h_stop);

    r.total_ms = (float) totalMs.count();


    // ── Bandwidth тооцоо ───────────────────────────────────────────────────
    // H2D: x болон y — 2 массив дамжуулсан
    r.h2d_bandwidth_gb = (2.0f * size) / (r.h2d_ms * 1e6f); // GB/s

    // D2H: result — 1 массив буцаасан
    r.d2h_bandwidth_gb = (1.0f * size) / (r.d2h_ms * 1e6f);

    // Kernel: 2 унших (x, y) + 1 бичих (result) = 3 × size
    r.kernel_bandwidth_gb = (3.0f * size) / (r.kernel_ms * 1e6f);


    // ── Хэвлэх ────────────────────────────────────────────────────────────
    printf("\n=== CUDA SAXPY Performance ===\n");
    printf("%-35s %8.3f ms\n", "H2D transfer time:",    r.h2d_ms);
    printf("%-35s %8.3f ms\n", "Kernel execution time:",  r.kernel_ms);
    printf("%-35s %8.3f ms\n", "D2H transfer time:",    r.d2h_ms);
    printf("%-35s %8.3f ms\n", "Total time (incl. malloc):", r.total_ms);
    printf("%-35s %8.2f GB/s\n", "H2D bandwidth:",         r.h2d_bandwidth_gb);
    printf("%-35s %8.2f GB/s\n", "D2H bandwidth:",         r.d2h_bandwidth_gb);
    printf("%-35s %8.2f GB/s\n", "Kernel effective bandwidth:", r.kernel_bandwidth_gb);
    printf("==============================\n");

    // ── Цэвэрлэх ──────────────────────────────────────────────────────────
    cudaFree(device_x);
    cudaFree(device_y);
    cudaFree(device_result);
    cudaEventDestroy(h2d_start); cudaEventDestroy(h2d_stop);
    cudaEventDestroy(k_start);   cudaEventDestroy(k_stop);
    cudaEventDestroy(d2h_start); cudaEventDestroy(d2h_stop);

    return r;
}

int main()
{
    int   N     = 1 << 20;
    float alpha = 2.0f;

    float *x      = (float *)malloc(N * sizeof(float));
    float *y      = (float *)malloc(N * sizeof(float));
    float *result = (float *)malloc(N * sizeof(float));

    for (int i = 0; i < N; i++) { x[i] = 1.0f; y[i] = 2.0f; }

    PerfResult perf = run_saxpy(N, alpha, x, y, result);


    bool success = true;
    for (int i = 0; i < 100; i++)
        if (result[i] != 4.0f) { success = false; break; }

    printf("Verification: %s\n", success ? "SUCCESS!" : "FAILED!");

    free(x); free(y); free(result);
    return 0;
}

Overwriting saxpy.cu


In [4]:
!nvidia-smi

Tue Apr 14 02:22:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
!nvcc saxpy.cu -o saxpy
!./saxpy

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

=== CUDA SAXPY Performance ===
H2D transfer time:                     2.097 ms
Kernel execution time:                 0.258 ms
D2H transfer time:                     2.816 ms
Total time (incl. malloc):           297.078 ms
H2D bandwidth:                          4.00 GB/s
D2H bandwidth:                          1.49 GB/s
Kernel effective bandwidth:            48.69 GB/s
Verification: SUCCESS!
